In [15]:
import requests
import pandas as pd

# Downloading data

### Downloading the Crop data

In [93]:
url = "https://analisi.transparenciacatalunya.cat/resource/yh94-j2n9.json"
params = {
    "$select": "campanya, municipi, cultiu, seca_regad, sum(ha) as ha_total",
    "$group": "campanya, municipi, cultiu, seca_regad",
    "$where": "campanya between '2016' and '2024'",
    "$limit": 1000000,
}
df_crops = pd.DataFrame(requests.get(url, params=params).json())
df_crops["ha_total"] = df_crops["ha_total"].astype(float)
print(len(df_crops)) # to check everything is downloaded

# Rename seca_regad
df_crops = df_crops.rename(columns={"seca_regad": "seca_regadiu"})
df_crops = df_crops[["campanya", "municipi", "cultiu", "seca_regadiu", "ha_total"]]
df_crops.head()

224307


,campanya,municipi,cultiu,seca_regadiu,ha_total
0,2023,VIMBODI I POBLET,GUARET / SUP. LLIURE SEMBRA,R,0.886906
1,2016,FORTIA,TRITICALE,S,9.260500
2,2020,ROCA DEL VALLES (LA),ESPECIES AROMÀTIQUES HERBÀCI,R,0.039900
3,2016,SANT JULIA DE RAMIS,BLAT DE MORO,R,42.190000
4,2020,ODENA,ERBS (LLEGUM) SIE,S,2.264700


### Downloading data of Daily meteorological data (XEMA) and Meteorological stations

In [90]:
url_daily = "https://analisi.transparenciacatalunya.cat/resource/7bvh-jvq2.json"
url_stations = "https://analisi.transparenciacatalunya.cat/resource/yqwd-vj5e.json"

# Daily meteorological observations
params_daily = {
    "$select": "Data_lectura, Codi_estacio, Codi_variable, valor",
    "$where": """
        Codi_variable IN ('1000', '1300')
        AND data_lectura >= '2016-01-01T00:00:00'
        AND data_lectura < '2025-01-01T00:00:00'
    """,
    "$limit": 100000000
}

# Station metadata
params_stations = {
    "$select": "codi_estacio, Nom_municipi",
    "$limit": 50000
}

df_daily = pd.DataFrame(requests.get(url_daily, params=params_daily).json())
df_stations = pd.DataFrame(requests.get(url_stations, params=params_stations).json())


print(len(df_daily))
print(len(df_stations))

1213106
245


### Merging and cleaning of Weather stations and lectures data

In [91]:
df_weather.head(5)
# We merge meteorological observations with station metadata
df_weather = df_daily.merge(
    df_stations[["codi_estacio", "Nom_municipi"]],
    left_on="Codi_estacio",
    right_on="codi_estacio",
    how="left"
)

# We convert date to dd/mm/yyyy (this is just my preference)
df_weather["Data_lectura"] = pd.to_datetime(
    df_weather["Data_lectura"],
    errors="coerce"
).dt.strftime("%d/%m/%Y")

# We create a Variable column
df_weather["Variable"] = df_weather["Codi_variable"].map({
    "1000": "Temperatura mitjana diària",
    "1300": "Precipitació acumulada diària",
    1000: "Temperatura mitjana diària",
    1300: "Precipitació acumulada diària"
})

# A renaming of the columns
df_weather = df_weather.rename(columns={
    "Data_lectura": "Data",
    "Codi_estacio": "Codi_estacio",
    "Nom_municipi": "Municipi",
    "Codi_variable": "Codi_variable",
    "valor": "Valor"
})

# Finally, we select and order columns
df_weather = df_weather[
    ["Data", "Codi_estacio", "Municipi",
     "Codi_variable", "Variable", "Valor"]
]

df_weather.head()

,Data,Codi_estacio,Municipi,Codi_variable,Variable,Valor
0,01/01/2016,C6,Castellnou de Seana,1000,Temperatura mitjana diària,6.7
1,01/01/2016,C6,Castellnou de Seana,1300,Precipitació acumulada diària,0
2,01/01/2016,C7,Tàrrega,1000,Temperatura mitjana diària,8.6
3,01/01/2016,C7,Tàrrega,1300,Precipitació acumulada diària,0.1
4,01/01/2016,C8,Cervera,1000,Temperatura mitjana diària,7.6


# Analysis and plotting

In [ ]:
We now have two different datasets, $\textit{df_weather}$, which has info of temperature and rain wherever there's a meteorologic station, and the $\textit{df_crops}$ which contains the 